##Step 1: Import libraries
Import libraries

In [0]:
import enum
import json
import time
import datetime
import gzip
import random
import requests
import traceback
import os
from urllib.parse import urlparse

##Step 2: Create client credentials secret scope
Store client credentials client in secret scope or set them explicitly. 

####Option1 - Creating a secret scope using Databricks CLI:
This option requires the ability to use the Databricks CLI in the workspace terminal or in command prompt.
1. In CONNECT data services, create a client credentials client and temporarily save the client id and secret generated. For this sample, the client credentials client needs to be given a role that has read and write access to a Cds namespace. For instructions, see:\
https://docs.aveva.com/bundle/connect-data-services/page/1263324.html
2. Create a secret scope called **cdsscope**. For instructions using the Databricks CLI, see:\
https://docs.databricks.com/en/security/secrets/index.html
3. Create two secrets within that secret scope called **cdsclientid** and **cdsclientsecret** containing the client id and secret generated from Cds. Refer to the link in step 2 for instructions.

####Option 2 - Creating a secret scope using Databricks SDK for Python:
This uses the Databricks SDK for Python option which requires entering your client credentials into the notebook in plain text. It's recommended to delete these credentials after the block is run.
1. In CONNECT data services, create a client credentials client and temporarily save the client id and secret generated. For this sample, the client credentials client needs to be given a role that has read and write access to a Cds namespace. For instructions, see:\
https://docs.aveva.com/bundle/connect-data-services/page/1263324.html
2. Enter your client credentials into the option 2 code block
3. Run the option 2 code block.

In [0]:
# use this to skip setting scope if already done
scopeAlreadyCreated = False

if scopeAlreadyCreated == False:
    
    from databricks.sdk import WorkspaceClient

    #enter your client id and secret here
    clientId = ""
    clientSecret = ""

    w = WorkspaceClient()

    #create scope
    try: 
        w.secrets.create_scope(scope='cdsscope')
    except Exception as e:
        print(e)

    #create secrets
    try:
        w.secrets.put_secret(scope='cdsscope', key='cdsclientid', string_value=clientId)
        w.secrets.put_secret(scope='cdsscope', key='cdsclientsecret', string_value=clientSecret)
    except Exception as e:
        print(e)

##Step 3: Get Token
Set connection information and use OAuth2.0 client credentials flow to get a bearer token.

In [0]:
#retrieve secrets from Databricks secrets
clientId = dbutils.secrets.get(scope = "cdsscope", key = "cdsclientid")
clientSecret = dbutils.secrets.get(scope = "cdsscope", key = "cdsclientsecret")

#set connection information
apiVersion = "v1"
resource = "https://uswe.datahub.connect.aveva.com" #change if using region other than US West
tenantId = ""
namespaceId = ""
omf_endpoint = f'{resource}/api/{apiVersion}/tenants/{tenantId}/namespaces/{namespaceId}/omf'
token_endpoint = f'{resource}/identity/connect/token'
webRequestTimeoutSeconds = 30

# OMF options
omf_version = '1.2'
useCompression = True

#set stream names
streamNames = ["databricks_stream_random1","databricks_stream_random2"]

# The number of seconds to sleep before sending another round of messages
sleep_time = 1

# Holders for data message values
boolean_value_1 = 0
boolean_value_2 = 1

#use the client ID and Secret to get the needed bearer token
token_information = requests.post(token_endpoint,data={'client_id': clientId,'client_secret': clientSecret,'grant_type': 'client_credentials'})
token = json.loads(token_information.content)["access_token"]
print("Bearer token is: " + token)

##Step 4: Define OMF Type and Container Messages
These can be modified to fit the data structure you would like to send to CONNECT data services.

NOTE: this script was designed using the v1.2
version of the OMF specification, as outlined here:
https://docs.aveva.com/bundle/omf/page/1283983.html 


In [0]:
omf_types = [
  {
    "id": "FirstDynamicType",
    "name": "First dynamic type",
    "classification": "dynamic",
    "type": "object",
    "description": "This is the first dynamic type",
    "properties": {
      "Timestamp": {
        "format": "date-time",
        "type": "string",
        "isindex": True,
        "description": "Timestamp property as index"
      },
      "IntegerProperty": {
        "type": "integer",
        "description": "PI point data referenced integer attribute",
        "uom": "count"
      }
    }
  },
  {
    "id": "SecondDynamicType",
    "name": "Second dynamic type",
    "classification": "dynamic",
    "type": "object",
    "description": "This is the second dynamic type",
    "properties": {
      "Timestamp": {
        "format": "date-time",
        "type": "string",
        "isindex": True,
        "description": "Timestamp property as index"
      },
      "NumberProperty1": {
        "type": "number",
        "description": "PI point data referenced number attribute 1",
        "format": "float64",
        "uom": "%"
      },
      "NumberProperty2": {
        "type": "number",
        "description": "PI point data referenced number attribute 2",
        "format": "float64",
        "uom": "V"
      },
      "StringEnum": {
        "type": "string",
        "enum": [ "False", "True" ],
        "description": "String enumeration to replace boolean type"
      }
    }
  },
  {
    "id": "ThirdDynamicType",
    "name": "Third dynamic type",
    "classification": "dynamic",
    "type": "object",
    "description": "This is the third dynamic type",
    "properties": {
      "Timestamp": {
        "format": "date-time",
        "type": "string",
        "isindex": True,
        "description": "Timestamp property as index"
      },
      "IntegerEnum": {
        "type": "integer",
        "format": "int16",
        "enum": [ 0, 1 ],
        "description": "Integer enumeration to replace boolean type"
      }
    }
  }
]

omf_containers = [
  {
    "id": "FirstContainer",
    "typeid": "FirstDynamicType"
  },
  {
    "id": "SecondContainer",
    "typeid": "FirstDynamicType"
  },
  {
    "id": "ThirdContainer",
    "typeid": "SecondDynamicType"
  },
  {
    "id": "FourthContainer",
    "typeid": "ThirdDynamicType"
  }
]

omf_data = [
  {
    "containerid": "FirstContainer",
    "values": [
      {
        "Timestamp": None,
        "IntegerProperty": None
      }
    ]
  },
  {
    "containerid": "SecondContainer",
    "values": [
      {
        "Timestamp": None,
        "IntegerProperty": None
      }
    ]
  },
  {
    "containerid": "ThirdContainer",
    "values": [
      {
        "Timestamp": None,
        "NumberProperty1": None,
        "NumberProperty2": None,
        "StringEnum": None
      }
    ]
  },
  {
    "containerid": "FourthContainer",
    "values": [
      {
        "Timestamp": None,
        "IntegerEnum": None
      }
    ]
  }
]

##Step 5: Set-up Functions
Run this next block to set up some utility functions for later.\
The **get_data** call should be customized to populate omf_data with relevant data. If you modified the **Type** and **Container** message structure in [Step 4](https://adb-4992701514070192.12.azuredatabricks.net/editor/notebooks/3664531415801454?o=4992701514070192#command/4756983619500680), you will need to modify **get_data** accordingly.

In [0]:

# ************************************************************************
# Wrapper function for sending an HTTP message
# ************************************************************************

def send_message_to_omf_endpoint(message_type, message_omf_json, action='create'):
    '''Sends the request out to the preconfigured endpoint'''

    # Compress json omf payload, if specified
    compression = 'none'
    if useCompression:
        msg_body = gzip.compress(bytes(json.dumps(message_omf_json), 'utf-8'))
        compression = 'gzip'
    else:
        msg_body = json.dumps(message_omf_json)

    # Collect the message headers
    msg_headers = get_headers(compression, message_type, action)

    # Send message to OMF endpoint
    response = {}
    response = requests.post(
        omf_endpoint,
        headers=msg_headers,
        data=msg_body,
        verify=True,
        timeout=webRequestTimeoutSeconds
    )

    # Check for 409, which indicates that a type with the specified ID and version already exists.
    if response.status_code == 409:
        return

    # response code in 200s if the request was successful!
    if response.status_code < 200 or response.status_code >= 300:
        print(msg_headers)
        response.close()
        print(
            f'Response from relay was bad. {message_type} message: {response.status_code} {response.text}.  Message holdings: {message_omf_json}')
        print()
        raise Exception(f'OMF message was unsuccessful, {message_type}. {response.status_code}:{response.text}')

# ************************************************************************
# Retrieves headers for HTTP request to the specified endpoint
# ************************************************************************

def get_headers(compression='', message_type='', action=''):
    '''Assemble headers for sending to the endpoint's OMF endpoint'''

    msg_headers = {
        'messagetype': message_type,
        'action': action,
        'messageformat': 'JSON',
        'omfversion': omf_version
    }

    if(compression == 'gzip'):
        msg_headers["compression"] = 'gzip'

    msg_headers["Authorization"] = f'Bearer {token}'

    # validate headers to prevent injection attacks
    validated_headers = {}

    for key in msg_headers:
        if key in {'Authorization', 'messagetype', 'action', 'messageformat', 'omfversion', 'x-requested-with', 'compression'}:
            validated_headers[key] = msg_headers[key]

    return validated_headers

# ************************************************************************
# This function will need to be customized to populate the OMF data
# message passed.
# ************************************************************************
def get_data(data):
    global boolean_value_1, boolean_value_2

    if data["containerid"] == 'FirstContainer' or data["containerid"] == 'SecondContainer':
        data["values"][0]["Timestamp"] = get_current_time()
        data["values"][0]["IntegerProperty"] = int(100*random.random())

    elif data["containerid"] == 'ThirdContainer':
        boolean_value_2 = (boolean_value_2 + 1) % 2
        data["values"][0]["Timestamp"] = get_current_time()
        data["values"][0]["NumberProperty1"] = 100*random.random()
        data["values"][0]["NumberProperty2"] = 100*random.random()
        data["values"][0]["StringEnum"] = str(bool(boolean_value_2))

    elif data["containerid"] == 'FourthContainer':
        boolean_value_1 = (boolean_value_1 + 1) % 2
        data["values"][0]["Timestamp"] = get_current_time()
        data["values"][0]["IntegerEnum"] = boolean_value_1

    else:
        print(f'Container {data["containerid"]} not recognized')

    return data


def get_current_time():
    ''' Returns the current time'''
    return datetime.datetime.utcnow().isoformat() + 'Z'


In [0]:
def main(last_sent_values={}):
    success = True

    try:
        # Send OMF Types
        for omf_type in omf_types:
            send_message_to_omf_endpoint('type', [omf_type])

        # Send OMF Containers
        for omf_container in omf_containers:
            send_message_to_omf_endpoint('container', [omf_container])

        # Send OMF Data
        count = 0
        # send data to all endpoints forever if this is not a test
        while count < 2:

            '''This is where custom loop logic should go.'''

            for omf_datum in omf_data:
                data_to_send = get_data(omf_datum)
                # send the data
                send_message_to_omf_endpoint('data', [data_to_send])

            time.sleep(sleep_time)
            count = count + 1

    except Exception as ex:
        print(f'Encountered Error: {ex}')
        print
        traceback.print_exc()
        print
        success = False

    print('Done')
    return success


if __name__ == '__main__':
    main()

##Step 6: Tests
Run this block after running the above blocks to test.

In [0]:
import unittest
from unittest.mock import patch, MagicMock

class TestOMFNotebook(unittest.TestCase):

    def test_get_current_time_format(self):
        t = get_current_time()
        self.assertTrue(t.endswith('Z'))
        self.assertIn('T', t)

    def test_get_data_firstcontainer(self):
        data = {
            "containerid": "FirstContainer",
            "values": [{"Timestamp": None, "IntegerProperty": None}]
        }
        result = get_data(data.copy())
        self.assertIsNotNone(result["values"][0]["Timestamp"])
        self.assertIsInstance(result["values"][0]["IntegerProperty"], int)

    def test_get_data_secondcontainer(self):
        data = {
            "containerid": "SecondContainer",
            "values": [{"Timestamp": None, "IntegerProperty": None}]
        }
        result = get_data(data.copy())
        self.assertIsNotNone(result["values"][0]["Timestamp"])
        self.assertIsInstance(result["values"][0]["IntegerProperty"], int)

    def test_get_data_thirdcontainer(self):
        data = {
            "containerid": "ThirdContainer",
            "values": [{"Timestamp": None, "NumberProperty1": None, "NumberProperty2": None, "StringEnum": None}]
        }
        result = get_data(data.copy())
        self.assertIsNotNone(result["values"][0]["Timestamp"])
        self.assertIsInstance(result["values"][0]["NumberProperty1"], float)
        self.assertIsInstance(result["values"][0]["NumberProperty2"], float)
        self.assertIn(result["values"][0]["StringEnum"], ["True", "False"])

    def test_get_data_fourthcontainer(self):
        data = {
            "containerid": "FourthContainer",
            "values": [{"Timestamp": None, "IntegerEnum": None}]
        }
        result = get_data(data.copy())
        self.assertIsNotNone(result["values"][0]["Timestamp"])
        self.assertIn(result["values"][0]["IntegerEnum"], [0, 1])

    def test_get_headers_required_keys(self):
        headers = get_headers('gzip', 'type', 'create')
        self.assertIn('Authorization', headers)
        self.assertIn('messagetype', headers)
        self.assertIn('action', headers)
        self.assertIn('messageformat', headers)
        self.assertIn('omfversion', headers)
        self.assertIn('compression', headers)
        self.assertEqual(headers['compression'], 'gzip')

    @patch('requests.post')
    def test_send_message_to_omf_endpoint_success(self, mock_post):
        mock_response = MagicMock()
        mock_response.status_code = 200
        mock_post.return_value = mock_response
        try:
            send_message_to_omf_endpoint('type', [omf_types[0]])
        except Exception:
            self.fail("send_message_to_omf_endpoint raised Exception unexpectedly!")

    @patch('requests.post')
    def test_send_message_to_omf_endpoint_409(self, mock_post):
        mock_response = MagicMock()
        mock_response.status_code = 409
        mock_post.return_value = mock_response
        # Should not raise
        send_message_to_omf_endpoint('type', [omf_types[0]])

    @patch('requests.post')
    def test_send_message_to_omf_endpoint_failure(self, mock_post):
        mock_response = MagicMock()
        mock_response.status_code = 400
        mock_response.text = "Bad Request"
        mock_post.return_value = mock_response
        with self.assertRaises(Exception):
            send_message_to_omf_endpoint('type', [omf_types[0]])

    @patch('time.sleep', return_value=None)
    @patch('requests.post')
    def test_main_success(self, mock_post, mock_sleep):
        mock_response = MagicMock()
        mock_response.status_code = 200
        mock_post.return_value = mock_response
        result = main()
        self.assertTrue(result)

    def test_cds_client_types_streams_data(self):
        # Use query param to filter by name for types and streams
        for t in omf_types:
            types_url = f"{resource}/api/{apiVersion}/tenants/{tenantId}/namespaces/{namespaceId}/types?query=id:{t['id']}"
            resp_types = requests.get(types_url, headers={"Authorization": f"Bearer {token}"})
            self.assertEqual(resp_types.status_code, 200)
            type_ids = [type_obj["Id"] for type_obj in resp_types.json()]
            self.assertIn(t["id"], type_ids)

        for c in omf_containers:
            streams_url = f"{resource}/api/{apiVersion}/tenants/{tenantId}/namespaces/{namespaceId}/streams?query=id:{c['id']}"
            resp_streams = requests.get(streams_url, headers={"Authorization": f"Bearer {token}"})
            self.assertEqual(resp_streams.status_code, 200)
            stream_ids = [stream_obj["Id"] for stream_obj in resp_streams.json()]
            self.assertIn(c["id"], stream_ids)

        # Check data for first container
        data_url = f"{resource}/api/{apiVersion}/tenants/{tenantId}/namespaces/{namespaceId}/streams/{omf_containers[0]['id']}/data/last"
        resp_data = requests.get(data_url, headers={"Authorization": f"Bearer {token}"})
        self.assertEqual(resp_data.status_code, 200)
        data = resp_data.json()
        self.assertTrue("Timestamp" in data and "IntegerProperty" in data)

if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)